# Itchy — Full Model on H100/A100

**Runtime > Change runtime type > H100 GPU (or A100)**, then **Runtime > Run all**

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install -q torch numpy huggingface-hub sentencepiece

import os, shutil, math, time, numpy as np
from pathlib import Path
from huggingface_hub import hf_hub_download
import sentencepiece as spm
import torch
import torch.nn.functional as F
from torch import nn

assert torch.cuda.is_available(), 'No GPU!'
device = torch.device('cuda')
gpu = torch.cuda.get_device_name(0)
mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu} ({mem:.0f}GB)')

In [ ]:
# ============================================================
# DOWNLOAD AND CONVERT DATA
# ============================================================
os.makedirs('data/bytes', exist_ok=True)

def dl(fn, sub, dst):
    if Path(dst).exists():
        print(f'  {dst} exists')
        return
    print(f'  Downloading {fn}...')
    c = str(Path(hf_hub_download(repo_id='willdepueoai/parameter-golf',
        filename=fn, subfolder=sub, repo_type='dataset')).resolve())
    try:
        os.link(c, dst)
    except OSError:
        shutil.copy2(c, dst)
    print(f'  -> {dst} ({Path(dst).stat().st_size/1e6:.0f}MB)')

dl('fineweb_train_000000.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/train0.bin')
dl('fineweb_train_000001.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/train1.bin')
dl('fineweb_val_000000.bin', 'datasets/datasets/fineweb10B_sp1024', 'data/val.bin')
dl('fineweb_1024_bpe.model', 'datasets/tokenizers', 'data/tok.model')

def load_shard(p):
    h = np.fromfile(p, dtype='<i4', count=256)
    return np.fromfile(p, dtype='<u2', count=int(h[2]), offset=256*4)

def write_shard(p, d):
    h = np.zeros(256, dtype='<i4')
    h[0] = 20240520; h[1] = 1; h[2] = len(d)
    with open(p, 'wb') as f:
        f.write(h.tobytes())
        f.write(d.astype('<u2').tobytes())

sp = spm.SentencePieceProcessor(model_file='data/tok.model')

for src, dst in [('data/train0.bin','data/bytes/train0.bin'),
                 ('data/train1.bin','data/bytes/train1.bin'),
                 ('data/val.bin','data/bytes/val.bin')]:
    if Path(dst).exists():
        print(f'  {dst} exists')
        continue
    print(f'  Converting {src}...')
    tok = load_shard(src)
    bs = [np.frombuffer(sp.decode(tok[i:i+100000].tolist()).encode('utf-8'), dtype=np.uint8)
          for i in range(0, len(tok), 100000)]
    write_shard(dst, np.concatenate(bs))
    print(f'    done')

def load_bytes(p):
    h = np.fromfile(p, dtype='<i4', count=256)
    return np.fromfile(p, dtype='<u2', count=int(h[2]), offset=256*4).astype(np.int64)

train_data = np.concatenate([load_bytes('data/bytes/train0.bin'), load_bytes('data/bytes/train1.bin')])
val_data = load_bytes('data/bytes/val.bin')
print(f'\nTrain: {len(train_data):,} bytes ({len(train_data)/1e6:.0f}MB)')
print(f'Val: {len(val_data):,} bytes ({len(val_data)/1e6:.0f}MB)')

In [ ]:
# ============================================================
# MODEL
# ============================================================

class Rotary(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer('inv_freq', inv_freq, persistent=False)
        self._cache = None
    def forward(self, seq_len, device, dtype):
        if self._cache is None or self._cache[0] != seq_len:
            t = torch.arange(seq_len, device=device, dtype=self.inv_freq.dtype)
            freqs = torch.outer(t, self.inv_freq.to(device))
            self._cache = (seq_len, freqs.cos()[None,None,:,:], freqs.sin()[None,None,:,:])
        return self._cache[1].to(dtype), self._cache[2].to(dtype)

def apply_rope(x, cos, sin):
    h = x.size(-1) // 2
    x1, x2 = x[..., :h], x[..., h:]
    return torch.cat((x1*cos + x2*sin, x1*(-sin) + x2*cos), dim=-1)

class Attn(nn.Module):
    def __init__(self, dim, nh, nkv, qk_gain):
        super().__init__()
        self.nh, self.nkv, self.hd = nh, nkv, dim // nh
        kv = nkv * self.hd
        self.cq = nn.Linear(dim, dim, bias=False)
        self.ck = nn.Linear(dim, kv, bias=False)
        self.cv = nn.Linear(dim, kv, bias=False)
        self.proj = nn.Linear(dim, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
        self.qg = nn.Parameter(torch.full((nh,), qk_gain))
        self.rot = Rotary(self.hd)
    def forward(self, x):
        B, S, D = x.shape
        q = self.cq(x).reshape(B,S,self.nh,self.hd).transpose(1,2)
        k = self.ck(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        v = self.cv(x).reshape(B,S,self.nkv,self.hd).transpose(1,2)
        q, k = F.rms_norm(q,(self.hd,)), F.rms_norm(k,(self.hd,))
        cos, sin = self.rot(S, x.device, q.dtype)
        q, k = apply_rope(q,cos,sin), apply_rope(k,cos,sin)
        q = q * self.qg.to(q.dtype)[None,:,None,None]
        y = F.scaled_dot_product_attention(q,k,v, is_causal=True, enable_gqa=(self.nkv!=self.nh))
        return self.proj(y.transpose(1,2).contiguous().reshape(B,S,D))

class MLP(nn.Module):
    def __init__(self, dim, mult):
        super().__init__()
        self.fc = nn.Linear(dim, dim*mult, bias=False)
        self.proj = nn.Linear(dim*mult, dim, bias=False)
        nn.init.zeros_(self.proj.weight)
    def forward(self, x):
        return self.proj(F.leaky_relu(self.fc(x), 0.5).square())

class Block(nn.Module):
    def __init__(self, dim, nh, nkv, mult, qk_gain):
        super().__init__()
        self.attn = Attn(dim, nh, nkv, qk_gain)
        self.mlp = MLP(dim, mult)
        self.as_ = nn.Parameter(torch.ones(dim))
        self.ms = nn.Parameter(torch.ones(dim))
        self.rm = nn.Parameter(torch.stack((torch.ones(dim), torch.zeros(dim))))
    def forward(self, x, x0):
        m = self.rm.to(x.dtype)
        x = m[0][None,None,:]*x + m[1][None,None,:]*x0
        x = x + self.as_.to(x.dtype)[None,None,:] * self.attn(F.rms_norm(x,(x.size(-1),)))
        x = x + self.ms.to(x.dtype)[None,None,:] * self.mlp(F.rms_norm(x,(x.size(-1),)))
        return x

class PerPositionDecode(nn.Module):
    def __init__(self, dim, patch_size, head_dim=128, vocab=260):
        super().__init__()
        self.ps = patch_size
        self.pos = nn.Parameter(torch.randn(patch_size, dim) * 0.02)
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(dim, head_dim, bias=False),
                nn.ReLU(),
                nn.Linear(head_dim, vocab, bias=False),
            )
            for _ in range(patch_size)
        ])
    def forward(self, x):
        logits = []
        for p in range(self.ps):
            logits.append(self.heads[p](x + self.pos[p][None,None,:]))
        return torch.stack(logits, dim=2)

class Itchy(nn.Module):
    def __init__(self, dim, num_layers, nh, nkv, mlp_mult, patch_size, head_dim=128, softcap=30.0):
        super().__init__()
        self.dim, self.ps, self.sc = dim, patch_size, softcap
        self.be = nn.Embedding(260, dim)
        self.pp = nn.Linear(dim * patch_size, dim, bias=False)
        ne = num_layers // 2
        nd = num_layers - ne
        self.ne, self.nd = ne, nd
        self.sw = nn.Parameter(torch.ones(min(ne, nd), dim))
        self.blocks = nn.ModuleList([Block(dim, nh, nkv, mlp_mult, 1.5) for _ in range(num_layers)])
        self.decode = PerPositionDecode(dim, patch_size, head_dim)

    def forward(self, ids, tgt):
        B, S = ids.shape
        P = self.ps
        x = self.be(ids).reshape(B, S//P, P * self.dim)
        x = F.rms_norm(self.pp(x), (self.dim,))
        x0 = x
        skips = []
        for i in range(self.ne):
            x = self.blocks[i](x, x0)
            skips.append(x)
        for i in range(self.nd):
            if skips:
                x = x + self.sw[i].to(x.dtype)[None,None,:] * skips.pop()
            x = self.blocks[self.ne + i](x, x0)
        x = F.rms_norm(x, (x.size(-1),))
        lo = self.decode(x).reshape(-1, 260)
        lo = self.sc * torch.tanh(lo / self.sc)
        return F.cross_entropy(lo.float(), tgt.reshape(-1))

n_params = lambda m: sum(p.numel() for p in m.parameters())
print('Model defined')

In [ ]:
# ============================================================
# CONFIG
# ============================================================

DIM = 384
NUM_LAYERS = 11
NUM_HEADS = 8
NUM_KV_HEADS = 4
MLP_MULT = 3
PATCH_SIZE = 12
DECODE_HEAD_DIM = 128
SEQ_LEN = 1020           # divisible by 12

TRAIN_STEPS = 7000
BATCH_SIZE = 65536
LR = 3e-4
LOG_EVERY = 500
VAL_EVERY = 2000
EMA_DECAY = 0.997

print(f'Config: dim={DIM} layers={NUM_LAYERS} mlp={MLP_MULT}x patch={PATCH_SIZE}')
print(f'        seq={SEQ_LEN} batch={BATCH_SIZE} steps={TRAIN_STEPS} decode_head={DECODE_HEAD_DIM}')

In [ ]:
# ============================================================
# TRAIN
# ============================================================

torch.manual_seed(1337)
model = Itchy(DIM, NUM_LAYERS, NUM_HEADS, NUM_KV_HEADS, MLP_MULT, PATCH_SIZE, DECODE_HEAD_DIM).to(device).bfloat16()
print(f'Itchy Final: {n_params(model):,} params ({n_params(model)/1e6:.1f}M)')
print(f'Int6 size: ~{n_params(model)*0.75/1e6:.1f}MB (budget: 16.0MB)')

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.95), weight_decay=0.01)
ema_state = {n: t.detach().float().clone() for n, t in model.state_dict().items()}

def run_val(use_ema=False):
    if use_ema:
        orig = {n: t.clone() for n, t in model.state_dict().items()}
        model.load_state_dict({n: t.to(orig[n].dtype) for n, t in ema_state.items()}, strict=True)
    losses = []
    with torch.no_grad():
        for i in range(0, min(500 * SEQ_LEN, len(val_data) - SEQ_LEN - 1), SEQ_LEN):
            chunk = val_data[i:i + SEQ_LEN + 1]
            x = torch.tensor(chunk[:-1].reshape(1, -1), device=device)
            y = torch.tensor(chunk[1:].reshape(1, -1), device=device)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                losses.append(model(x, y).item())
    if use_ema:
        model.load_state_dict(orig, strict=True)
    return np.mean(losses) / math.log(2)

pos = 0
t0 = time.time()
losses = []

print(f'\nTraining {TRAIN_STEPS} steps with EMA on {gpu}...\n')

for step in range(1, TRAIN_STEPS + 1):
    usable = (BATCH_SIZE // SEQ_LEN) * SEQ_LEN
    end = pos + usable + 1
    if end > len(train_data):
        pos = 0
        end = usable + 1
    chunk = train_data[pos:end]
    pos = end - 1
    x = torch.tensor(chunk[:-1].reshape(-1, SEQ_LEN), device=device)
    y = torch.tensor(chunk[1:].reshape(-1, SEQ_LEN), device=device)

    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        loss = model(x, y)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

    with torch.no_grad():
        for n, t in model.state_dict().items():
            ema_state[n].mul_(EMA_DECAY).add_(t.detach().float(), alpha=1.0 - EMA_DECAY)

    if step % LOG_EVERY == 0 or step == 1:
        avg = np.mean(losses[-LOG_EVERY:])
        bpb = avg / math.log(2)
        elapsed = time.time() - t0
        eta = elapsed / step * (TRAIN_STEPS - step)
        print(f'step {step:>5} | loss {avg:.4f} | bpb {bpb:.4f} | '
              f'{elapsed/60:.1f}min | ~{eta/60:.0f}min left')

    if VAL_EVERY > 0 and step % VAL_EVERY == 0:
        vb = run_val(use_ema=False)
        vb_ema = run_val(use_ema=True)
        print(f'  >>> VAL step {step}: {vb:.4f} BPB (raw) | {vb_ema:.4f} BPB (EMA)')

total_time = time.time() - t0
print(f'\nTraining done in {total_time/60:.1f} min')

In [ ]:
# ============================================================
# FINAL VALIDATION
# ============================================================

print('='*70)
print('FINAL VALIDATION')
print('='*70)

def full_val(use_ema=False):
    if use_ema:
        orig = {n: t.clone() for n, t in model.state_dict().items()}
        model.load_state_dict({n: t.to(orig[n].dtype) for n, t in ema_state.items()}, strict=True)
    losses = []
    with torch.no_grad():
        for i in range(0, min(2000 * SEQ_LEN, len(val_data) - SEQ_LEN - 1), SEQ_LEN):
            chunk = val_data[i:i + SEQ_LEN + 1]
            x = torch.tensor(chunk[:-1].reshape(1, -1), device=device)
            y = torch.tensor(chunk[1:].reshape(1, -1), device=device)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                losses.append(model(x, y).item())
    if use_ema:
        model.load_state_dict(orig, strict=True)
    val_loss = np.mean(losses)
    return val_loss, val_loss / math.log(2)

raw_loss, raw_bpb = full_val(use_ema=False)
ema_loss, ema_bpb = full_val(use_ema=True)

best_bpb = min(raw_bpb, ema_bpb)

print(f'\n  Raw model:  val_loss={raw_loss:.4f}  val_bpb={raw_bpb:.4f}')
print(f'  EMA model:  val_loss={ema_loss:.4f}  val_bpb={ema_bpb:.4f}')
print(f'\n  Model: {n_params(model):,} params ({n_params(model)/1e6:.1f}M)')
print(f'  Int6 size: ~{n_params(model)*0.75/1e6:.1f}MB')
print(f'  Training: {total_time/60:.1f} min on {gpu}')
print(f'  Data: {len(train_data)/1e6:.0f}MB train, {len(val_data)/1e6:.0f}MB val (2 shards)')
print()
print(f'  Competition reference (8xH100, 10min, 80 shards):')
print(f'    Baseline:    1.2244 BPB')
print(f'    SOTA merged: 1.1194 BPB')
print(f'    Itchy:       {best_bpb:.4f} BPB  <---')
print()
if best_bpb < 1.12:
    print('  >>> BEATS COMPETITION SOTA (on 2 shards, not directly comparable)')
elif best_bpb < 1.22:
    print('  >>> BEATS BASELINE (on 2 shards, not directly comparable)')
else:
    print('  >>> Below baseline — may improve with more data/steps')
print()
print('  NOTE: Competition uses 80 shards + sliding window + TTT.')
print('  This uses 2 shards + simple eval. Real score will differ.')